## This script is written for the purpose of analysising PositioNZ and private CORS data. 
In order to analyse these data, it uses the PositioNZ network as a benchamrk that individual CORS can be tested against, have outliers removed, count the number of sites that are removed in this process and calcualte the percentage of sites from each private network that are removed. 

In [ ]:
import requests
import pandas as pd
import numpy as np
import datetime
from astropy.time import Time
from LINZ.CORS_Timeseries import TimeseriesList, CoordApiTimeseries
import os 
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf
import matplotlib.pyplot as plt
from scipy.signal import spectrogram, detrend

The timeframe can be set to any length, as long as there is data present.

In [ ]:
# Define the timeframe
start = datetime.datetime(2025, 4, 10)
end = datetime.datetime(2025, 12, 3)

#### All of the CORS sites are listed as four letter codes, these can be easily added when new sites are installed/updated down the line. 

There is a need to differentiate between the PositioNZ and private CORS sites as the PositioNZ CORS will be used to set a benchmark all of the other private CORS will be compared to.

In [ ]:
code_mappings = {
    "PNcodes": "PositoNZ",
    "Vcodes": "Trimble",
    "PPcodes": "Position_Partners",
    "EScodes": "Elliot_sinclair",
    "GScodes": "Global_Survey",
    "SYcodes": "Synergy",
    "SHcodes": "Survey_Hire"
}

# Print out the PositioNZ code to ensure it has worked
print("PNcodes =", code_mappings["PNcodes"])


The PositioNZ CORS are entered here.

In [ ]:
# The PositioNZ CORS codes are printed here: 

# PositioNZ CORS
PNcodes=['AUCK','BLUF','CHTI','CORM','DNVK','DUND','GISB','GLDB','HAAS','HAMT','HAST','HIKB','HOKI','KAIK','KTIA','LEXA','LKTA','MAHO','MAVL','METH','MQZG','MTJO','NLSN','NPLY','PYGR','TAUP','TRNG',
         'VGMT','WAIM','WANG','WARK','WEST','WGTN','WHKT','WHNG','WRPA']

All of the other private CORS are entered here. These have been set up in a manner to allow new sites to be easily added.

In [ ]:
# The Trimble V codes are printed here:

Vcodes=['VAK2',	'VALE',	'VCHR',	'VHAM',	'VHN2',	'VHRA',	'VLEE',	'VMAY',	'VNPE',	'VOMA',	'VOTG',	'VOTO',	'VPOA', 'VPRK',	'VPUK',	'VTEA',	'VTEK','VTEM', 'VWAN', 'VWKU', 'VWPK', 'VZQN','VOMA','VMKA',
        'VSHF','VTES','VTOK','VWK2','VWM2','VPN2']


# The Position Partners codes are printed here: 
PPcodes=['PPCW','PPNL','PPOR','PPPK','PPRS','PPSP','PPK2']
#Inactive codes=['PPMK']


# The Elliot Sinclair codes are printed here: 
EScodes=['ESBR','ESGM','ESKV','ESNL','ESPM','ESQT','ESRN','ESTR']
#Inactive codes=['ESCM']

# The Global Survey codes are printed here: 
GScodes=['GSAA','GSAB','GSAM','GSAY','GSBA','GSBD','GSCC','GSCT','GSDA','GSDR','GSGR','GSHT','GSHU','GSHW','GSIN','GSKA','GSKK','GSLI','GSLW', 'GSLX','GSMG','GSOM','GSOK','GSPE','GSPL','GSPM','GSPN',
         'GSPO','GSPR','GSPU','GSQT','GSRA','GSRL','GSRO','GSSB','GSTA','GSTH','GSTK','GSTU','GSTW','GSUH','GSUT','GSWI','GSWR','GSWU','GSNP','GSHO','GSGL','GSBN','GSBR','GSMA','GSMD','GSMO','GSPK',
         'GSWA','GSMH','GSTI','GSWH']

# The Synergy codes are printed here: 
SYcodes=['SYAS','SYCC','SYCM','SYDF','SYHA','SYHW','SYM2','SYMV','SYNL','SYQN','SYT2','SYTA','SYTS','SYWM','SYET','SYBM','SYRD','SYLN','SYWK','SYPP','SYP2', 'SYOI', 'SYP3', 'SYPO', 'SYPR']

# Survey Hire codes are printed here: 
SHcodes=['SHAR','SHMD','SHPA','SHML','SHSA']

The PositioNZ CORS data and private CORS data are downloaded using the solution type: p20f_54_code

In [ ]:
# Calculate crd_epoch
crd_epoch = end + (start - end) / 2
astropy_time_object = Time(crd_epoch, format='datetime')
crd_epoch_decimal_year = astropy_time_object.decimalyear

# Initialise a dictionary to store DataFrames
daily_avg_dfs = {}

# Loop through PNcodes
for code in PNcodes:
    try:
        print(f"Processing station: {code}")
        ts = CoordApiTimeseries(code, solutiontype='p20f_54_code', after=start, before=end)  
        dates, xyz = ts.withoutOutliers().getObs(enu=False)

        df_obs = pd.DataFrame(xyz, columns=['x', 'y', 'z'], index=dates)
        df_obs['date'] = df_obs.index.strftime('%Y-%m-%d')
        df_obs['station'] = code

        daily_avg_dfs[code] = df_obs

    except Exception as e:
        print(f"Failed to process station {code} with error: {e}")

# Combine all station DataFrames into one
PN_data = pd.concat(daily_avg_dfs.values(), axis=0)

# Sort alphabetically by station and date
PN_data.sort_values(by=["station", "date"], inplace=True)

# Print confirmation
print("Download complete ✅")
print("PN_data DataFrame created with shape:", PN_data.shape)
print("Columns:", PN_data.columns.tolist())


In [ ]:
# Initialise tracking set
stations_with_no_data = set()

# Dictionary to store group DataFrames
group_dataframes = {}

# Loop through all groups except PNcodes
for group_label, group_name in code_mappings.items():
    if group_label == "PNcodes":
        continue  # Skip PNcodes (already processed)

    print(f"\nProcessing group: {group_label} ({group_name})")

    # Get the actual list of station codes from the variable name
    station_list = globals().get(group_label)
    if not station_list:
        print(f"⚠️ No station list found for {group_label}")
        continue

    daily_avg_dfs = {}

    for code in station_list:
        try:
            print(f"  Processing station: {code}")
            ts = CoordApiTimeseries(code, solutiontype='p20f_54_code', after=start, before=end)
            dates, xyz = ts.withoutOutliers().getObs(enu=False)

            df_obs = pd.DataFrame(xyz, columns=['x', 'y', 'z'], index=dates)
            df_obs['date'] = df_obs.index.strftime('%Y-%m-%d')
            df_obs['station'] = code

            daily_avg_dfs[code] = df_obs

        except Exception as e:
            print(f"    Failed to process station {code} with error: {e}")
            stations_with_no_data.add(code)

    if daily_avg_dfs:
        sorted_codes = sorted(daily_avg_dfs.keys())
        group_df = pd.concat([daily_avg_dfs[code] for code in sorted_codes], axis=0)
        group_df.sort_values(by=["station", "date"], inplace=True)

        df_name = group_label.replace("codes", "_data")
        group_dataframes[df_name] = group_df

        print("Download complete ✅")
        print(f"{df_name} DataFrame created with shape: {group_df.shape}")
        print(f"Columns: {group_df.columns.tolist()}")

    else:
        print(f"⚠️ No data collected for group: {group_label}")


In [ ]:
group_dataframes = {"PN_data": PN_data, **{k: v for k, v in group_dataframes.items() if k != "PN_data"}}

#### Check that all of the dataframes have printed correctly 

In [ ]:
print("Current group DataFrames:")
for df_name in group_dataframes.keys():
    print(f" - {df_name}")

#### Define the dataframes that the converted coordinates will fill

In [ ]:
# Define the columns expected after coordinate conversion
converted_columns = [
    'x', 'y', 'z', 'date', 'station',
    'nztm_easting', 'nztm_northing', 'nzvd2016_elev'
]

# Prepare empty DataFrames to hold converted coordinates for each group
empty_converted_dfs = {
    f"{name}_converted": pd.DataFrame(columns=converted_columns)
    for name in group_dataframes.keys()
}

# Optional: Preview one of the empty DataFrames
print(empty_converted_dfs["PN_data_converted"].head())


Convert the data in the dataframes from the ITRF coordinate system, to NZTM2000/NZVD2016

In [ ]:
converted_group_dataframes = {}

def convert_coordinates(input_crds, crd_epoch_decimal_year):
    occ_api = "https://www.geodesy.linz.govt.nz/api/conversions/v1/convert-to"
    coordlist = {
        "crs": "LINZ:ITRF2020_XYZ",
        "coordinateOrder": ["geocentricX", "geocentricY", "geocentricZ"],
        "coordinateEpoch": crd_epoch_decimal_year,
        "coordinates": input_crds
    }
    params = {"crs": "LINZ:NZTM/NZVD2016"}

    response = requests.post(occ_api, params=params, json=coordlist)
    if response.status_code == 200:
        converted = response.json()
        return converted['coordinateList']['coordinates']
    else:
        print(f"❌ API conversion failed with status {response.status_code}")
        print(f"🔍 Error details: {response.text}")
        return []



In [ ]:
for name, original_df in group_dataframes.items():
    print(f"\n🔄 Processing DataFrame: {name}")

    # Always start with a fresh copy
    df = original_df.copy(deep=True)

    # Identify and print NaN values
    nan_values = df[df.isna().any(axis=1)]
    if not nan_values.empty:
        print(f" - Found {len(nan_values)} NaN rows")

    # Identify and print Infinity values
    inf_values = df[(df == np.inf) | (df == -np.inf)].dropna(how='all')
    if not inf_values.empty:
        print(f" - Found {len(inf_values)} Inf rows")

    # Filter out invalid rows
    df_filtered = df.dropna(subset=['x', 'y', 'z']).copy()
    df_filtered = df_filtered[
        (df_filtered['x'] != np.inf) & (df_filtered['x'] != -np.inf) &
        (df_filtered['y'] != np.inf) & (df_filtered['y'] != -np.inf) &
        (df_filtered['z'] != np.inf) & (df_filtered['z'] != -np.inf)
    ]

    if df_filtered.empty:
        print(f"⚠️ Skipping {name} — no valid coordinates after filtering.")
        continue

    # Prepare coordinates
    input_crds = df_filtered[['x', 'y', 'z']].values.tolist()

    # Convert coordinates
    converted_coords = convert_coordinates(input_crds, crd_epoch_decimal_year)

    # Add converted coordinates to DataFrame
    if converted_coords:
        df_filtered[['nztm_easting', 'nztm_northing', 'nzvd2016_elev']] = pd.DataFrame(
            converted_coords, index=df_filtered.index
        )

        # Store in the pre-initialised DataFrame
        converted_name = f"{name}_converted"
        if converted_name in empty_converted_dfs:
            empty_converted_dfs[converted_name] = df_filtered
            print(f"✅ {converted_name} filled and updated.")
        else:
            print(f"⚠️ {converted_name} not found in empty_converted_dfs. Skipping storage.")
    else:
        print(f"❌ Conversion failed for {name}. No coordinates added.")


In [ ]:
# Rename for clarity — this is now the final dictionary of converted DataFrames
converted_group_dataframes = empty_converted_dfs
del empty_converted_dfs  # Optional: clean up the old name if no longer needed


In [ ]:

# Loop through each network
for network_name, df in converted_group_dataframes.items():
    if df.empty or 'nzvd2016_elev' not in df.columns or 'date' not in df.columns or 'station' not in df.columns:
        continue

    df['date'] = pd.to_datetime(df['date'])

    # Group by station
    for station_name, station_df in df.groupby('station'):
        if station_df.empty:
            continue

        station_df = station_df.sort_values('date')
        station_df.set_index('date', inplace=True)

        # Apply running mean
        station_df['running_mean'] = station_df['nzvd2016_elev'].rolling(f'{window_days}D', min_periods=1).mean()

        # Detrend by subtracting the running mean and convert to millimeters
        station_df['detrended_mm'] = (station_df['nzvd2016_elev'] - station_df['running_mean']) * 1000

        # Plot running mean and detrended residuals together
        fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

        # Top subplot: original and running mean
        ax[0].plot(station_df.index, station_df['nzvd2016_elev'], label='Original Elevation', alpha=0.6)
        ax[0].plot(station_df.index, station_df['running_mean'], label=f'Running Mean ({window_days} Days)', color='orange')
        ax[0].set_title(f'Elevation and Running Mean - {network_name} / {station_name}')
        ax[0].set_ylabel('Elevation (m)')
        ax[0].legend()
        ax[0].grid(True)

        # Bottom subplot: detrended residuals in millimeters
        ax[1].plot(station_df.index, station_df['detrended_mm'], label='Detrended Residuals (mm)', color='purple')
        ax[1].axhline(0, color='gray', linestyle='--', linewidth=1)
        ax[1].set_title(f'Detrended Residuals - {network_name} / {station_name}')
        ax[1].set_xlabel('Date')
        ax[1].set_ylabel('Residuals (mm)')
        ax[1].legend()
        ax[1].grid(True)

        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


In [ ]:


station_metrics = []

for network_name, df in converted_group_dataframes.items():
    if df.empty or 'nzvd2016_elev' not in df.columns or 'date' not in df.columns or 'station' not in df.columns:
        continue

    df['date'] = pd.to_datetime(df['date'])

    for station_name, station_df in df.groupby('station'):
        if station_df.empty or station_name == 'SYBM':
            continue

        station_df = station_df.sort_values('date')
        station_df.set_index('date', inplace=True)

        window_days = (datetime.datetime(2025, 5, 30) - datetime.datetime(2025, 4, 10)).days
        station_df['running_mean'] = station_df['nzvd2016_elev'].rolling(f'{window_days}D', min_periods=1).mean()
        station_df['detrended_mm'] = (station_df['nzvd2016_elev'] - station_df['running_mean']) * 1000

        noise_std = station_df['detrended_mm'].std()
        acf_values = acf(station_df['detrended_mm'].dropna(), nlags=20, fft=True)

        metrics = {
            'network': network_name,
            'station': station_name,
            'noise_std_mm': noise_std,
            'acf_lag_1': acf_values[1] if len(acf_values) > 1 else None,
            'acf_lag_5': acf_values[5] if len(acf_values) > 5 else None,
            'acf_lag_10': acf_values[10] if len(acf_values) > 10 else None
        }

        station_metrics.append(metrics)

# Create the final DataFrame
metrics_df = pd.DataFrame(station_metrics)



In [ ]:


station_metrics = []

for network_name, df in converted_group_dataframes.items():
    if df.empty or 'nzvd2016_elev' not in df.columns or 'date' not in df.columns or 'station' not in df.columns:
        continue

    df['date'] = pd.to_datetime(df['date'])

    for station_name, station_df in df.groupby('station'):
        if station_df.empty:
            continue  # SYBM is no longer excluded

        station_df = station_df.sort_values('date')
        station_df.set_index('date', inplace=True)

        window_days = (datetime.datetime(2025, 5, 30) - datetime.datetime(2024, 4, 10)).days
        station_df['running_mean'] = station_df['nzvd2016_elev'].rolling(f'{window_days}D', min_periods=1).mean()
        station_df['detrended_mm'] = (station_df['nzvd2016_elev'] - station_df['running_mean']) * 1000

        noise_std = station_df['detrended_mm'].std()
        acf_values = acf(station_df['detrended_mm'].dropna(), nlags=20, fft=True)

        station_metrics.append({
            'network': network_name,
            'station': station_name,
            'noise_std_mm': noise_std,
            'acf_lag_1': acf_values[1] if len(acf_values) > 1 else None,
            'acf_lag_5': acf_values[5] if len(acf_values) > 5 else None,
            'acf_lag_10': acf_values[10] if len(acf_values) > 10 else None
        })

metrics_df = pd.DataFrame(station_metrics)

# Summary statistics
network_summary = metrics_df.groupby('network')['noise_std_mm'].agg(['mean', 'median', 'std', 'count']).reset_index()
print("Noise Summary by Network:")
print(network_summary)

# Bar plot
plt.figure(figsize=(10, 6))
plt.bar(network_summary['network'], network_summary['median'], color='lightblue')
plt.ylabel('Median Noise Level (mm)')
plt.title('Median Final Acquisition GNSS Noise Level by Network')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y')
plt.tight_layout()
plt.show()


# Box plot
plt.figure(figsize=(10, 6))
metrics_df.boxplot(column='noise_std_mm', by='network', grid=False)
plt.ylabel('Noise Level (mm)')
plt.title('GNSS Noise Distribution by Network')
plt.suptitle('')
plt.xlabel('Network')

# Rotate and offset x-axis labels
ax = plt.gca()
for label in ax.get_xticklabels():
    label.set_rotation(45)
    label.set_ha('right')
    x, y = label.get_position()
    label.set_position((x - 0.1, y)) 

plt.tight_layout()
plt.show()





In [ ]:

# Initialise list to collect station metrics
station_metrics = []

# Loop through each network and station
for network_name, df in converted_group_dataframes.items():
    if df.empty or 'nzvd2016_elev' not in df.columns or 'date' not in df.columns or 'station' not in df.columns:
        continue

    df['date'] = pd.to_datetime(df['date'])

    for station_name, station_df in df.groupby('station'):
        if station_df.empty or station_name == 'SYBM':
            continue

        station_df = station_df.sort_values('date')
        station_df.set_index('date', inplace=True)

        # Compute running mean and detrended residuals
        station_df['running_mean'] = station_df['nzvd2016_elev'].rolling(f'{window_days}D', min_periods=1).mean()
        station_df['detrended_mm'] = (station_df['nzvd2016_elev'] - station_df['running_mean']) * 1000

        # Compute statistics
        detrended = station_df['detrended_mm'].dropna()
        noise_std = detrended.std()
        noise_mean = detrended.mean()
        noise_median = detrended.median()

        station_metrics.append({
            'station': station_name,
            'network': network_name,
            'mean_detrended_mm': noise_mean,
            'median_detrended_mm': noise_median,
            'std_detrended_mm': noise_std
        })

# Convert to DataFrame and save
station_stats_df = pd.DataFrame(station_metrics)
station_stats_df.to_csv('PP_station_detrended_statistics.csv', index=False)

# Print summary
print("Mean, Median, and Standard Deviation of Detrended Residuals per Station:")
print(station_stats_df)


In [ ]:
# Calculate mean, median, and standard deviation of noise level per station
station_noise_stats = metrics_df.groupby('station')['noise_std_mm'].agg(['mean', 'median']).reset_index()

# Display the results
print("Mean, Median")
print(station_noise_stats)



In [ ]:

# Calculate overall mean and median of noise level
overall_mean_noise = station_stats_df['std_detrended_mm'].mean()
overall_median_noise = station_stats_df['std_detrended_mm'].median()


# Print per-station statistics
print("\nPer-Station Noise Statistics:")
print(station_stats_df[['station', 'std_detrended_mm']])

# Convert to DataFrame and save
station_stats_df = pd.DataFrame(station_metrics)
station_stats_df.to_csv('New_PP_station_detrended_statistics.csv', index=False)
